In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Dict, Any
tqdm.pandas()
sys.path.append("../../../")
plt.rcParams["font.size"] = 7

from src.data.data_splits import generate_split_mask

In [ ]:
def get_info(dataframe:pd.DataFrame):
    return len(dataframe), dataframe.BDSPPatientID.nunique()

In [ ]:
data_path = Path("../../../data/raw/heedb")
diag_df = pd.read_csv(data_path / "12SL_diagnoses/diagnoses.csv", low_memory=False)
meta_df = pd.read_csv(data_path / "metadata/metadata.csv", low_memory=False)

In [ ]:
diag_dict = pd.read_csv(data_path / "12SL_diagnoses/diagnoses_dictionary.csv", low_memory=False)

In [ ]:
diag_df

In [ ]:
meta_df

In [ ]:
list(meta_df.columns)

In [ ]:
diag_dict

In [ ]:
# remove all '*' and ';' characters from the 'diagnoses' column (this can cause isssues for correctly matching diagnoses and their corresponding ICD codes)
misplaced_chars = ['*', ';', "(", ")", ":", "?"]
for char in misplaced_chars:
    diag_dict['diagnoses'] = diag_dict['diagnoses'].str.replace(char, '', regex=False)
diag_dict['diagnoses'] = diag_dict['diagnoses'].str.strip() 

In [ ]:
diag_dict

In [ ]:
from src.data.constants import HEEDB_CODE_DICT
HEEDB_CODE_DICT

In [ ]:
# there are some incosistencies (duplicates) in the assigned ICD codes: 
#   'nonspecific st abnormality': 900 & 1021,
#   'atrial flutter': 162 & 273,
#   'supraventricular tachycardia': 266 & 271,
#   'sinus bradycardia': 21 & 24
duplicate_icds = [(900, 1021), (162, 273), (1116, 1117), (173, 181), (266, 271), (21, 24)]

In [ ]:
# overwrite duplicate icd codes
# this is an inefficient solution, but one that makes sure no substrings are accidentally matched
def overwrite_duplicates(code_string:str, duplicate_list:List, verbose:bool=False)->str:
    codes = code_string.split(',')
    codes = [int(code.strip()) for code in codes]
    for code, dupe in duplicate_list:
        if dupe in codes:
            print(f"Overwriting duplicate ICD code {dupe} with {code} in codes {codes}") if verbose else None
            codes.remove(dupe)
            codes.append(code)
    codes = [str(code) for code in codes]
    return ','.join(codes)

diag_df['codes'] = diag_df['codes'].progress_apply(lambda x: overwrite_duplicates(x, duplicate_icds))

In [ ]:
# standardize FileName format
print(meta_df.iloc[0].FileName, diag_df.iloc[0].FileName)
diag_df.FileName = diag_df.FileName.str.replace(".hea", "")
diag_df.FileName = diag_df.FileName.str.replace("\n", "")
diag_df.FileName = diag_df.FileName.str[1:]
print(meta_df.iloc[0].FileName, diag_df.iloc[0].FileName)

In [ ]:
# merge dataframes
df = pd.merge(diag_df, meta_df, on='FileName', how='inner')
print(f"{len(df)} ECGs from {df.BDSPPatientID.nunique()} patients.")

In [ ]:
df.head()

## ECGs per patient (after filtering)

In [ ]:
import scipy
plt.figure(figsize=(2.5, 1.75))
counts = df.groupby('BDSPPatientID').size().values
mean_count = np.mean(counts)
median_count = np.median(counts)
percentile = np.percentile(counts, 90)
# empirical survival function of counts
res = scipy.stats.ecdf(counts)
res.sf.plot(color="blue")
plt.axvline(mean_count, color='red', label=f"Mean: {mean_count:.2f}")
plt.axvline(median_count, color='green', label=f"Median: {median_count}")
plt.axvline(percentile, color='orange', label=f"90th perc: {percentile}")
plt.xlabel("Number of ECGs per Patient")
plt.ylabel("1 - Cumulative Probability")
plt.legend()
plt.yscale("log")
plt.title("HEEDB: #ECGs per Patient")
plt.savefig("../../../figs/heedb/heedb_record_stats.pdf", bbox_inches='tight')

### Create column for binary (multi-hot) prediction label

In [ ]:
def write_binary_labels(row:pd.Series, verbose:bool=False):
    codes = [int(code.strip()) for code in row['codes'].split(',')]
    matched_codes = []
    for c in codes:
        if c in HEEDB_CODE_DICT.keys():
            row[f'icd_{c}'] = True
            matched_codes.append(c)
    print(f"matched {len(matched_codes)}/{len(codes)} icd codes") if verbose else None
    if len(matched_codes) != len(codes) and verbose:
        print("available codes:", codes)
        print("matched codes:", matched_codes)
    return row

In [ ]:
for icd_code in HEEDB_CODE_DICT.keys():
    df[f'icd_{icd_code}'] = False 
    df[f'icd_{icd_code}'].astype('bool')
df = df.progress_apply(write_binary_labels, axis=1)

In [ ]:
def print_icd_occurence_counts(dataframe:pd.DataFrame, icd_code_dict:Dict[int, str]):
    print("ICD Code Occurrence Counts:")
    for code, name in icd_code_dict.items():
        count = dataframe[f'icd_{code}'].sum()
        print(f"{name} ({code}): {count}")

In [ ]:
print_icd_occurence_counts(df, HEEDB_CODE_DICT)

## Dataset Splits

In [ ]:
df.ECGAcquisitionTime = pd.to_datetime(df.ECGAcquisitionTime, errors='coerce')
# drop any nan dates
nan_dates = df.ECGAcquisitionTime.isna().sum()
if nan_dates > 0:
    print(f"Dropping {nan_dates} rows with invalid ECGAcquisitionTime")
    df = df[~df.ECGAcquisitionTime.isna()]
# convert dateofbirth to datetime
df.DateOfBirth = pd.to_datetime(df.DateOfBirth, errors='coerce')
# replace 'redacted' with NaT
df.DateOfBirth = df.DateOfBirth.replace('redacted', pd.NaT)

In [ ]:
# standardise columns with other datasets and fix age
df = df.rename(columns={'SexDSC': 'sex'})
df['age'] = df.ECGAcquisitionTime.dt.year - df.DateOfBirth.dt.year

In [ ]:
df.age.plot.hist(figsize=(3,2))

In [ ]:
print(f"{len(df)} ECGs from {df.BDSPPatientID.nunique()} patients.")

In [ ]:
# patient-wise train-val-test split
val_test_pids = pd.Series(df.BDSPPatientID.unique()).sample(
    n=35_000, random_state=42, replace=False
)
val_pids = val_test_pids[:10_000]
test_pids = val_test_pids[10_000:]
traindf = df[~df.BDSPPatientID.isin(val_pids) & ~df.BDSPPatientID.isin(test_pids)]
val_df = df[df.BDSPPatientID.isin(val_pids)]
test_df = df[df.BDSPPatientID.isin(test_pids)]
traindf.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
print(f"Train (historical+future): {get_info(traindf)}")
print(f"Validation: {get_info(val_df)}")
print(f"Test: {get_info(test_df)}")


In [ ]:
# longitudinal (patient-specific) evaluation split
eval_mask = generate_split_mask(
    dataframe=traindf,
    patient_id_col="BDSPPatientID",
    timestamp_col="ECGAcquisitionTime",
    label_cols=[f'icd_{code}' for code in HEEDB_CODE_DICT.keys()],
    n_holdout_classes=2,
)   

train_df = traindf[~eval_mask]
long_eval_df = traindf[eval_mask]
train_df.reset_index(drop=True, inplace=True)
long_eval_df.reset_index(drop=True, inplace=True)
del traindf
print(f"Train: {get_info(train_df)}")
print(f"Longitudinal Evaluation: {get_info(long_eval_df)}")
print(f"Validation: {get_info(val_df)}")
print(f"Test: {get_info(test_df)}")

In [ ]:
eval_pids = pd.Series(long_eval_df.BDSPPatientID.unique())
assert sum(eval_pids.isin(train_df["BDSPPatientID"])) == len(
    eval_pids
), f"Every patient in longitudinal eval set should also be in train set: {sum(eval_pids.isin(train_df['BDSPPatientID']))} vs {len(eval_pids)}"

In [ ]:
# we create a helper dataframe that indicates all/any diseases present per patient in the training set
present_diseases_by_patient = train_df.groupby('BDSPPatientID')[[f"icd_{c}" for c in HEEDB_CODE_DICT.keys()]].max()
present_diseases_by_patient

In [ ]:
from typing import List

# we generate two new columns for each future record
    # 'seen_diseases': comma-separated list of disease names that were already present in the historical records of the same patient
    # 'unseen_diseases': comma-separated list of disease names that are present in the future record but were not present in the historical records of the same patient

def case_stratification_by_historical_disease_presence(row:pd.Series, historical_diseases_by_patient:pd.DataFrame, label_cols:List[str], patient_id_col:str="BDSPPatientID", verbose:bool=False) -> bool:
    pid = row[patient_id_col]
    label = row[label_cols]
    present_diseases_historical = historical_diseases_by_patient.loc[pid]
    assert present_diseases_historical.shape == label.shape, f"Shape mismatch: {present_diseases_historical.shape} vs {label.shape}"
    positive_unseen = label[(label == 1) & (present_diseases_historical == 0)]
    positive_seen = label[((label == 1) & (present_diseases_historical == 1)) | ((label == 0) & (present_diseases_historical == 0))]
    print(f"Patient {pid} has unseen diseases: {positive_unseen.index.tolist()}") if verbose and len(positive_unseen) > 0 else None
    print(f"Patient {pid} has seen diseases: {positive_seen.index.tolist()}") if verbose and len(positive_seen) > 0 else None
    unseen_labels = positive_unseen.index.tolist() if len(positive_unseen) > 0 else []
    seen_labels = positive_seen.index.tolist() if len(positive_seen) > 0 else []
    assert set(unseen_labels).isdisjoint(set(seen_labels)), f"Seen and unseen labels should be disjoint: {unseen_labels} vs {seen_labels}"
    unseen_labels_str = ",".join(unseen_labels)
    seen_labels_str = ",".join(seen_labels)
    return pd.Series([seen_labels_str, unseen_labels_str]) # must return pd.Series to fill two columns simultaneously

In [ ]:
i = np.random.randint(0, len(long_eval_df)-1)
case_stratification_by_historical_disease_presence(long_eval_df.iloc[i], present_diseases_by_patient, [f"icd_{c}" for c in HEEDB_CODE_DICT.keys()], verbose=True)

In [ ]:
long_eval_df[['seen_diseases', 'unseen_diseases']] = long_eval_df.progress_apply(
    lambda row: case_stratification_by_historical_disease_presence(
        row,
        historical_diseases_by_patient=present_diseases_by_patient,
        label_cols=[f"icd_{c}" for c in HEEDB_CODE_DICT.keys()],
        patient_id_col="BDSPPatientID",
        verbose=False
    ),
    axis=1,
    result_type='expand'
)
long_eval_df['has_unseen_disease'] = long_eval_df['unseen_diseases'].apply(lambda x: len(x) > 0)
long_eval_df['has_seen_disease'] = long_eval_df['seen_diseases'].apply(lambda x: len(x) > 0)

In [ ]:
long_eval_df.has_unseen_disease.value_counts()

In [ ]:
long_eval_df.unseen_diseases.value_counts()

In [ ]:
train_df[[f"icd_{c}" for c in HEEDB_CODE_DICT.keys()]].sum()

In [ ]:
test_df[[f"icd_{c}" for c in HEEDB_CODE_DICT.keys()]].sum()

In [ ]:
csv_path = Path("../../../data/csv")
train_df.to_pickle(csv_path / "heedb_train_historical.pkl")

In [ ]:
long_eval_df.to_pickle(csv_path / "heedb_train_future.pkl")

In [ ]:
val_df.to_pickle(csv_path / "heedb_val.pkl")

In [ ]:
test_df.to_pickle(csv_path / "heedb_test.pkl")

In [ ]:
print("Train:")
print_icd_occurence_counts(train_df, HEEDB_CODE_DICT)

In [ ]:
print("Test:")
print_icd_occurence_counts(test_df, HEEDB_CODE_DICT)